# GAM — Gated Associative Memory

A minimal demo running the GAM layer and comparing it with a same-sized Transformer on a copy task.

Run each cell in order.  The last cell will train both models and show results.

In [ ]:
!pip install torch numpy matplotlib -q
import math, time, sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
print("PyTorch", torch.__version__)

In [ ]:
# ── GAM Layer ────────────────────────────────────────────────────────
class GAMLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)
        self.Wg = nn.Linear(d, d, bias=True)
        self.ln1 = nn.LayerNorm(d)
        self.W1 = nn.Linear(d, d*4)
        self.W2 = nn.Linear(d*4, d)
        self._reset()
    def _reset(self):
        lim = 1.0 / math.sqrt(d)
        for w in [self.Wq, self.Wk, self.Wv, self.Wg]:
            nn.init.uniform_(w.weight, -lim, lim)
        nn.init.constant_(self.Wg.bias, -2.0)
        nn.init.xavier_uniform_(self.W1.weight)
        nn.init.xavier_uniform_(self.W2.weight)
    def forward(self, X):
        B, T, d = X.shape
        H = torch.zeros(B, d, d, device=X.device)
        outs = []
        for t in range(T):
            x = X[:, t, :]
            q = self.Wq(x); k = self.Wk(x); v = self.Wv(x)
            g = torch.sigmoid(self.Wg(x))
            read = torch.bmm(H, q.unsqueeze(-1)).squeeze(-1)
            err = v - torch.bmm(H, k.unsqueeze(-1)).squeeze(-1)
            H = H + g.unsqueeze(-1) * err.unsqueeze(-2) * k.unsqueeze(-1)
            out = self.ln1(read + x)
            z = F.gelu(self.W1(out))
            out = out + self.W2(z)
            outs.append(out.unsqueeze(1))
        return torch.cat(outs, dim=1)

# ── Transformer Layer (single-head, for fair comparison) ────────────
class TransformerLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)
        self.Wo = nn.Linear(d, d, bias=False)
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.W1 = nn.Linear(d, d*4); self.W2 = nn.Linear(d*4, d)
        self._reset()
    def _reset(self):
        lim = 1.0 / math.sqrt(d)
        for w in [self.Wq, self.Wk, self.Wv, self.Wo]:
            nn.init.uniform_(w.weight, -lim, lim)
        nn.init.xavier_uniform_(self.W1.weight)
        nn.init.xavier_uniform_(self.W2.weight)
    def forward(self, X):
        B, T, d = X.shape
        q = self.Wq(X); k = self.Wk(X); v = self.Wv(X)
        att = torch.bmm(q, k.transpose(1,2)) / math.sqrt(d)
        mask = torch.triu(torch.full((T,T), float('-inf')), diagonal=1).to(X.device)
        att = F.softmax(att + mask, dim=-1)
        out = self.Wo(torch.bmm(att, v))
        out = self.ln1(out + X)
        z = F.gelu(self.W1(out))
        return self.ln2(out + self.W2(z))

### Create data: copy task

Sequences of random tokens; the model must predict the next token (causal LM).
When the loss converges, the model has learned to copy.

In [ ]:
VOCAB = 32; D_MODEL = 64; BATCH = 32; EPOCHS = 30
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

def make_data(n):
    xs, ys = [], []
    for _ in range(n):
        L = np.random.randint(8, 25)
        seq = np.random.randint(0, VOCAB, size=L).tolist()
        xs.append([VOCAB] + seq)
        ys.append(seq + [VOCAB])
    return xs, ys

train_x, train_y = make_data(1000)
test_x,  test_y  = make_data(200)

def pad(xs, ys):
    mx = max(len(s) for s in xs)
    inp = torch.full((len(xs), mx), VOCAB, dtype=torch.long)
    tgt = torch.full((len(xs), mx), -100, dtype=torch.long)
    for i,(x,y) in enumerate(zip(xs,ys)):
        inp[i,:len(x)] = torch.tensor(x, dtype=torch.long)
        tgt[i,:len(y)] = torch.tensor(y, dtype=torch.long)
    return inp.to(DEVICE), tgt.to(DEVICE)

In [ ]:
# Build models
class SinusoidalPE(nn.Module):
    def __init__(self, d, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2, dtype=torch.float) * (-math.log(10000.0)/d))
        pe[:,0::2] = torch.sin(pos * div)
        pe[:,1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:,:x.size(1),:]

def build(variant):
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(VOCAB+1, D_MODEL)
            self.pe = SinusoidalPE(D_MODEL)
            self.layer = GAMLayer(D_MODEL) if variant=='gam' else TransformerLayer(D_MODEL)
            self.head = nn.Linear(D_MODEL, VOCAB+1)
            nn.init.normal_(self.embed.weight, std=0.02)
            nn.init.normal_(self.head.weight, std=0.02)
        def forward(self, x):
            return self.head(self.layer(self.pe(self.embed(x))))
    return M().to(DEVICE)

gam   = build('gam')
trans = build('trans')
print(f"GAM params: {sum(p.numel() for p in gam.parameters()):,}")
print(f"Transformer params: {sum(p.numel() for p in trans.parameters()):,}")

In [ ]:
# Training loop
def run(model, opt, data_x, data_y, train=True):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    idx = list(range(len(data_x)))
    if train: np.random.shuffle(idx)
    with torch.set_grad_enabled(train):
        for s in range(0, len(idx), BATCH):
            bi = idx[s:s+BATCH]
            inp, tgt = pad([data_x[i] for i in bi], [data_y[i] for i in bi])
            loss = F.cross_entropy(model(inp).reshape(-1, VOCAB+1), tgt.reshape(-1), ignore_index=-100)
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            total += loss.item(); n += 1
    return total / n

opt_gam   = torch.optim.AdamW(gam.parameters(),   lr=3e-4)
opt_trans = torch.optim.AdamW(trans.parameters(),  lr=3e-4)
h = {"g_t":[], "g_e":[], "t_t":[], "t_e":[]}

for ep in range(1, EPOCHS+1):
    h["g_t"].append(run(gam,   opt_gam,   train_x, train_y, True))
    h["t_t"].append(run(trans, opt_trans, train_x, train_y, True))
    h["g_e"].append(run(gam,   None, test_x, test_y, False))
    h["t_e"].append(run(trans, None, test_x, test_y, False))
    if ep % 10 == 0:
        print(f"ep {ep:3d} | GAM train {h['g_t'][-1]:.4f} test {h['g_e'][-1]:.4f} | Trans train {h['t_t'][-1]:.4f} test {h['t_e'][-1]:.4f}")

In [ ]:
# Final accuracy
def acc(model, data_x, data_y):
    model.eval()
    c, t = 0, 0
    with torch.no_grad():
        for s in range(0, len(data_x), BATCH):
            inp, tgt = pad(data_x[s:s+BATCH], data_y[s:s+BATCH])
            p = model(inp).argmax(-1)
            m = tgt != -100
            c += (p[m]==tgt[m]).sum().item()
            t += m.sum().item()
    return c/t

ag = acc(gam, test_x, test_y)
at = acc(trans, test_x, test_y)
print(f"\nTest accuracy — GAM {ag:.3f} | Transformer {at:.3f}")

# Plot
fig, ax = plt.subplots(1,2,figsize=(10,3.5))
ax[0].plot(h['g_t'], label='GAM train'); ax[0].plot(h['g_e'], label='GAM test')
ax[0].plot(h['t_t'], label='Trans train'); ax[0].plot(h['t_e'], label='Trans test')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend()
ax[1].bar(['GAM','Transformer'],[ag,at], color=['#4C72B0','#DD8452'])
ax[1].set_ylabel('Accuracy')
for bar,val in zip(ax[1].patches, [ag,at]):
    ax[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}',
               ha='center', va='bottom')
plt.tight_layout(); plt.show()

### Interpretation

Both models have nearly identical parameter counts (~54k).
GAM uses **O(n)** time, the Transformer uses **O(n²)**.
If accuracy is similar, GAM wins on efficiency.

See [github.com/FiveTechSoft/GAM](https://github.com/FiveTechSoft/GAM) for
the full architecture description, related work, and multi-layer benchmarks.